In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import time
import serial
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Instruments
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Instruments.VICI_valves.dim import DIM
from Instruments.VICI_valves.fsm import FSM
from Instruments.Syr_pumps.HB_syr_pump import HBElite
from Instruments.Knauer.knauer_pump_azura import KnauerPumpAzura

# Ecosystems
from Ecosystems.reaction_segment_generation import RSG
from Ecosystems.segmentation import Segmentation

# Logger
from Core.logging import flow_logger as logger

# Experiment framework
from Core.experiment_context import ExperimentManager
from Core.experiment_builder import ExperimentBuilder
from Core.experiment_validator import ExperimentValidator

In [2]:
# --- Syringe pump ---
syr_pump = HBElite("COM10")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- FSM ---
fsm = FSM("COM2")
fsm.connect()

# --- Carrier pump ---
k_pump = KnauerPumpAzura("COM4")
k_pump.connect()

# --- Gilson ---
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("Segmentation_testing")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=155.5,
    y_offset=10,
    x_min=145,
    x_max=250,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=319,
    y_offset=39,
    x_min=275,
    x_max=370,
    y_min=2,
    y_max=292
) 

tray.add_module(
    slot=3,
    name="dim",
    module=dim,
    x_offset=9,     #These values are to be changed
    y_offset=104,
    x_min=0,
    x_max=25,
    y_min=75,
    y_max=120
) 

# --- Ecosystems ---
rsg = RSG(gilson=g, pump=syr_pump, dim=dim)

seg = Segmentation(
    dim=dim,
    fsm=fsm,
    carrier_pump=k_pump,
    rsg=rsg
)

print("Hardware connected.")

HB Elite pump connected on COM10
Connected to DIM on COM5
Connected to FSM on COM2
Connected to Azura pump on COM4
[Pump] Flow stopped
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT
Hardware connected.


In [6]:
rows = [
    {
        "slug_id": "slug_1",
        "module": "rack2",
        "vial": 1,          # MeCN source
        "volume_uL": 100,
    }
]

builder = ExperimentBuilder()

result = builder.build_and_create(
    experiment_id="VB-1E-5",
    rows=rows,
    description="Single 100 uL MeCN slug commissioning test",
    global_conditions={
        "flowrate_ul_min": 1000,   # 1 mL/min
        "gas_prime_s": 20
    },
    overwrite=True
)

print(result["plan_path"])
pd.DataFrame(result["summary"])

C:\Users\CHAD-HPLC\Documents\VictorFlow\Experiments\VB-1E-5\plan.json


,slug_id,num_components,total_volume_uL,components
0,slug_1,1,100.0,rack2:1 (100.0 uL)


In [7]:
manager = ExperimentManager()

manager.load_experiment("VB-1E-5")

[EXPERIMENT] VB-1E-5 loaded
[EXPERIMENT] Plan ready
[EXPERIMENT] Log ready
[EXPERIMENT] State = prepared
[EXPERIMENT] Awaiting arm_experiment()


In [8]:
manager.precondition_reactor(seg, carrier_flowrate_ul_min=1000, stabilisation_time_s=60)

[REACTOR] Setting valve positions
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT
[k_pump] Flow rate set to 1000 uL/min
[Pump] Flow started
[REACTOR] Stabilising...
[SYSTEM] Reactor READY


In [9]:
manager.prepare_rsg(
    seg,
    air_gap=20.0,
    rate_wdr=0.10
)

manager.status()
print(seg.state)

[RSG] Initialising
[Initialise] Setting up start condition
[Air Gap] 20.0uL @ 0.1mL/min
[Initialise] Ready - air gap in place
[SYSTEM] RSG READY
Mode: experiment
Experiment: VB-1E-5
State: prepared
System state: UNKNOWN
Reactor state: REACTOR_READY
RSG state: RSG_READY
Platform state: PLATFORM_READY
Slug index: 0
Segmentation phase = READY | DIM = INJECT | FSM = CARRIER_TO_DIM


In [10]:
manager.arm_experiment()
manager.status()

[EXPERIMENT] VB-1E-5 armed
[EXPERIMENT] Awaiting execute_experiment()
Mode: experiment
Experiment: VB-1E-5
State: armed
System state: UNKNOWN
Reactor state: REACTOR_READY
RSG state: RSG_READY
Platform state: PLATFORM_READY
Slug index: 0


In [11]:
preview = manager.preview()
pd.DataFrame(preview)


[EXPERIMENT PREVIEW]
------------------------------------------------------------
1. slug_1 | 1 components | 100.0 uL
   rack2:1 (100.0 uL)
------------------------------------------------------------
[EXPERIMENT] 1 slugs total



,slug_number,slug_id,num_components,total_volume_uL,components
0,1,slug_1,1,100.0,rack2:1 (100.0 uL)


In [12]:
print("Confirm before running:")
print("- rack2 vial1 contains MeCN")
print("- waste vial available")
print("- nitrogen on")
print("- carrier solvent MeCN")
print("- tubing connected downstream")
print("- no leaks visible")
print("- hand ready for emergency stop")
print()
manager.status()
print(seg.state)

Confirm before running:
- rack2 vial1 contains MeCN
- waste vial available
- nitrogen on
- carrier solvent MeCN
- tubing connected downstream
- no leaks visible
- hand ready for emergency stop

Mode: experiment
Experiment: VB-1E-5
State: armed
System state: UNKNOWN
Reactor state: REACTOR_READY
RSG state: RSG_READY
Platform state: PLATFORM_READY
Slug index: 0
Segmentation phase = READY | DIM = INJECT | FSM = CARRIER_TO_DIM


In [13]:
results = manager.execute_experiment(seg)

results

[EXPERIMENT] Executing VB-1E-5
[FSM] Valve moved to A -> state = GAS_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT
[Segmentation] Phase: READY -> GAS_PRIMED
[Segmentation] Phase: GAS_PRIMED -> LOOP_LOADING
[DIM] Valve moved to B -> state = DIMState.LOAD
[Build Reaction Segment] slug_1
[Pickup] 100.0uL from rack2 vial 1 @ 0.05mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DIM] 100.0uL dispensed in DIM @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[Segmentation] Phase: LOOP_LOADING -> LOOP_LOADED
[DIM] Valve moved to A -> state = DIMState.INJECT
[FSM] Valve moved to B -> state = CARRIER_TO_DIM
[k_pump] Flow rate set to 1000.0 uL/min
[Pump] Flow started
[Segmentation] Phase: LOOP_LOADED -> RUNNING
[Segmentation] Segment launched at 1000.0 µL/min
[EXPERIMENT] Completed slug_1 (100.0 uL)
[EXPERIMENT] VB-1E-5 completed


[{'slug_id': 'slug_1',
  'dispensed_volume_ul': 100.0,
  'num_components': 1,
  'launched': True}]

In [ ]:
manager.status()
print(seg.state)

results

In [ ]:
import json
from pathlib import Path

log_path = Path("Experiments/VB-1E-5/log.json")

with open(log_path, "r") as f:
    log = json.load(f)

pd.DataFrame(log["events"])

In [14]:
k_pump.stop_flow()

[Pump] Flow stopped


'OFF:OK'